# Command로 상태 갱신과 이동

한 노드가 판정 결과를 상태에 기록하면서 다음 노드까지 함께 정할 수 있을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 상태와 Command 노드 정의

다음 코드에서 상태와 Command 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [2]:
# 상태와 Command 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

class DecisionState(TypedDict):
    score: float
    decision: str
    answer: str

def route(state: DecisionState) -> Command[Literal['generate', 'request_more']]:
    enough = state['score'] >= 0.7
    return Command(
        update={'decision': 'sufficient' if enough else 'insufficient'},
        goto='generate' if enough else 'request_more',
    )

## 목적지 노드와 그래프 정의

다음 코드에서 목적지 노드와 그래프 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [3]:
# 목적지 노드와 그래프 정의에 사용할 상태 형식과 함수를 선언합니다.
def generate(state: DecisionState) -> dict:
    return {'answer': '선택한 근거로 답변을 생성합니다.'}

def request_more(state: DecisionState) -> dict:
    return {'answer': '답변 전에 자료를 추가로 확인합니다.'}

builder = StateGraph(DecisionState)
builder.add_node('route', route)
builder.add_node('generate', generate)
builder.add_node('request_more', request_more)
builder.add_edge(START, 'route')
builder.add_edge('generate', END)
builder.add_edge('request_more', END)
graph = builder.compile()

## 두 경로 실행

두 경로 실행 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [4]:
# 두 경로 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
base = {'decision': '', 'answer': ''}
high = graph.invoke({**base, 'score': 0.82})
low = graph.invoke({**base, 'score': 0.41})
print('0.82:', high['decision'], high['answer'])
print('0.41:', low['decision'], low['answer'])
assert high['decision'] == 'sufficient'
assert low['decision'] == 'insufficient'

0.82: sufficient 선택한 근거로 답변을 생성합니다.
0.41: insufficient 답변 전에 자료를 추가로 확인합니다.


## 가능한 이동 시각화

컴파일한 그래프에서 Mermaid 문자열을 생성합니다. 출력에서 노드 이름과 연결 방향을 확인합니다.

In [5]:
# 컴파일된 그래프의 실제 노드와 엣지를 Mermaid 문자열로 확인합니다.
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)
assert 'generate' in mermaid and 'request_more' in mermaid

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	generate(generate)
	request_more(request_more)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	route -.-> generate;
	route -.-> request_more;
	generate --> __end__;
	request_more --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

